In [ ]:
import MODE_utilities as AS

import os
import gc
import random
import time

import numpy as np
import pandas as pd
import torch

In [ ]:
# ============================================================
# MODE settings - FIXED for all scalability experiments
# ============================================================

POPULATION = 20
OFFSPR = 10
GENS = 10

GAMMA = 1.0
NORMALIZE = True

MIN_CLUSTER_SIZE = 2
USE_DIVERGENCE = False

# Internal train/validation split over the barycenter representation
SPLIT = 0.80

MAX_ITER = 50
BARYCENTER_MAX_ITER = 50
D_STAR_TP = 1.8

ASF_WEIGHTS_FULL = np.array(
    [0.6, 0.2, 0.2]
)

TRACK_ENERGY = True


# ============================================================
# External offline / online split
# ============================================================

OFFLINE_FRACTION = 0.80


# ============================================================
# Dataset
# "cpu" or "memory"
# ============================================================

TYPE_OF_DATA = "cpu"


# ============================================================
# Reproducibility
# ============================================================

SEED = 89


# ============================================================
# Output
# ============================================================

OUTPUT_ROOT = os.path.join(
    "MODE_scalability_results",
    TYPE_OF_DATA,
)

os.makedirs(
    OUTPUT_ROOT,
    exist_ok=True,
)

In [ ]:
def set_all_seeds(seed=SEED):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_all_seeds()

print("Device:", AS.device)
print("Data type:", TYPE_OF_DATA)

In [ ]:
def evaluate_online_with_context(
    model_instance,
    offline_series,
    online_series,
    normalized=True,
    eps=1e-8,
):
    """
    Evaluate one online horizon using the final points of the offline
    horizon as forecasting context.

    Predictions/targets correspond ONLY to online observations.

    This permits short online horizons, e.g.:
        full length 50 -> online length 10
    even if the selected LSTM window is up to 20.
    """

    offline_series = np.asarray(
        offline_series,
        dtype=np.float64,
    ).ravel()

    online_series = np.asarray(
        online_series,
        dtype=np.float64,
    ).ravel()

    window = int(
        model_instance.chromosome["window"]
    )

    if len(offline_series) < window:
        raise ValueError(
            f"Offline series length {len(offline_series)} "
            f"is smaller than selected window {window}."
        )

    # Historical context only
    context = offline_series[-window:]

    # Evaluation sequence:
    # [last offline context | complete online horizon]
    evaluation_series = np.concatenate(
        [
            context,
            online_series,
        ]
    )

    # ------------------------------------------------------------
    # Normalize using the ONLINE horizon's range
    # ------------------------------------------------------------

    if normalized:

        s_min = online_series.min()
        s_max = online_series.max()
        s_range = s_max - s_min

        evaluation_used = (
            evaluation_series - s_min
        ) / (
            s_range + eps
        )

    else:

        s_min = None
        s_range = None

        evaluation_used = (
            evaluation_series.copy()
        )

    # ------------------------------------------------------------
    # Create exactly one prediction per online observation
    # ------------------------------------------------------------

    X = []
    y = []

    for i in range(
        len(evaluation_used) - window
    ):

        X.append(
            evaluation_used[
                i:i + window
            ]
        )

        y.append(
            evaluation_used[
                i + window
            ]
        )

    X = np.asarray(
        X,
        dtype=np.float32,
    )

    y = np.asarray(
        y,
        dtype=np.float32,
    )

    X_tensor = (
        torch.tensor(
            X,
            dtype=torch.float32,
        )
        .unsqueeze(-1)
        .to(AS.device)
    )

    # ------------------------------------------------------------
    # Predict
    # ------------------------------------------------------------

    model_instance.model = (
        model_instance.model.to(
            AS.device
        )
    )

    model_instance.model.eval()

    with torch.inference_mode():

        preds_used = (
            model_instance.model(
                X_tensor
            )
            .cpu()
            .numpy()
            .flatten()
        )

    targets_used = y.flatten()

    # Sanity check:
    # one prediction for every online target
    assert len(preds_used) == len(
        online_series
    )

    # ------------------------------------------------------------
    # Normalized RMSE
    # ------------------------------------------------------------

    rmse_used = float(
        np.sqrt(
            np.mean(
                (
                    preds_used
                    - targets_used
                ) ** 2
            )
        )
    )

    # ------------------------------------------------------------
    # Back to raw scale
    # ------------------------------------------------------------

    if normalized:

        preds_raw = (
            preds_used
            * (s_range + eps)
            + s_min
        )

        targets_raw = (
            targets_used
            * (s_range + eps)
            + s_min
        )

    else:

        preds_raw = preds_used
        targets_raw = targets_used

    rmse_raw = float(
        np.sqrt(
            np.mean(
                (
                    preds_raw
                    - targets_raw
                ) ** 2
            )
        )
    )

    raw_range = (
        targets_raw.max()
        - targets_raw.min()
    )

    nrmse_raw = float(
        100.0
        * rmse_raw
        / (raw_range + eps)
    )

    return {
        "rmse_normalized":
            rmse_used,

        "rmse_raw":
            rmse_raw,

        "nrmse_raw":
            nrmse_raw,

        "predictions_raw":
            preds_raw,

        "targets_raw":
            targets_raw,

        "window":
            window,
    }

In [ ]:
def run_mode_scalability_case(
    split_data,
    experiment_part,
    scale_value,
    full_series_length,
    output_folder,
):

    set_all_seeds(SEED)

    os.makedirs(
        output_folder,
        exist_ok=True,
    )

    split_data = np.asarray(
        split_data,
        dtype=np.float64,
    )

    if split_data.ndim != 2:

        raise ValueError(
            f"Expected 2-D time-series array, "
            f"received {split_data.shape}."
        )

    n_series = split_data.shape[0]

    # ============================================================
    # Offline / online
    # ============================================================

    offline_length = int(
        OFFLINE_FRACTION
        * full_series_length
    )

    offline_data = split_data[
        :, :offline_length
    ]

    online_data = split_data[
        :, offline_length:
    ]

    print("\n" + "=" * 80)
    print(
        f"PART {experiment_part} | "
        f"Scale value = {scale_value}"
    )
    print("=" * 80)

    print(
        "Number of time series:",
        n_series,
    )

    print(
        "Full series length:",
        full_series_length,
    )

    print(
        "Offline shape:",
        offline_data.shape,
    )

    print(
        "Online shape:",
        online_data.shape,
    )

    print(
        "Total offline values:",
        offline_data.size,
    )

    # ============================================================
    # MODE
    # ============================================================

    initial_output, initial_energy = (
        AS.run_with_energy_tracking(

            func=lambda: AS.ApplyingMODE(

                data=offline_data,

                num_population=POPULATION,

                num_offsprings=OFFSPR,

                num_generations=GENS,

                gamma=GAMMA,

                max_iter=MAX_ITER,

                barycenter_max_iter=(
                    BARYCENTER_MAX_ITER
                ),

                normalize=NORMALIZE,

                use_divergence=(
                    USE_DIVERGENCE
                ),

                min_cluster_size=(
                    MIN_CLUSTER_SIZE
                ),

                split=SPLIT,

                D_star_tp=D_STAR_TP,
            ),

            label=(
                f"MODE_scalability_"
                f"{experiment_part}_"
                f"{scale_value}"
            ),

            enable=TRACK_ENERGY,
        )
    )

    (
        initial_results,
        initial_problem,
        initial_duration,
    ) = initial_output

    timing = initial_problem.timing

    # ============================================================
    # ASF final solution
    # ============================================================

    (
        initial_instance,
        initial_model,
        best_chromosome,
        initial_info,
    ) = AS.ApplyingASF(

        results_of_tuning=(
            initial_results
        ),

        Problem_class=(
            initial_problem
        ),

        weights=(
            ASF_WEIGHTS_FULL
        ),

        return_full=True,
    )

    # ============================================================
    # Evaluate ALL online time series
    # ============================================================

    online_results = []

    for series_id in range(
        n_series
    ):

        res = evaluate_online_with_context(

            model_instance=(
                initial_instance
            ),

            offline_series=(
                offline_data[
                    series_id
                ]
            ),

            online_series=(
                online_data[
                    series_id
                ]
            ),

            normalized=NORMALIZE,
        )

        online_results.append({

            "series_id":
                int(series_id),

            "rmse_normalized":
                res[
                    "rmse_normalized"
                ],

            "rmse_raw":
                res[
                    "rmse_raw"
                ],

            "nrmse_raw":
                res[
                    "nrmse_raw"
                ],
        })

    raw_rmse_values = np.asarray(
        [
            row["rmse_raw"]
            for row in online_results
        ],
        dtype=float,
    )

    normalized_rmse_values = np.asarray(
        [
            row["rmse_normalized"]
            for row in online_results
        ],
        dtype=float,
    )

    # ============================================================
    # TXT output
    # ============================================================

    file_name = (
        f"MODE_Part_{experiment_part}_"
        f"{scale_value}.txt"
    )

    txt_path = os.path.join(
        output_folder,
        file_name,
    )

    with open(
        txt_path,
        "w",
        encoding="utf-8",
    ) as f:

        f.write(
            "MODE SCALABILITY EXPERIMENT\n"
        )

        f.write("=" * 72 + "\n")

        f.write(
            f"Part: {experiment_part}\n"
        )

        f.write(
            f"Scale value: {scale_value}\n"
        )

        f.write(
            f"Resource type: "
            f"{TYPE_OF_DATA}\n"
        )

        # ========================================================
        # Dataset information
        # ========================================================

        f.write(
            "\nDATASET\n"
        )

        f.write("-" * 72 + "\n")

        f.write(
            f"Number of time series: "
            f"{n_series}\n"
        )

        f.write(
            f"Full series length: "
            f"{full_series_length}\n"
        )

        f.write(
            f"Offline series length: "
            f"{offline_data.shape[1]}\n"
        )

        f.write(
            f"Online series length: "
            f"{online_data.shape[1]}\n"
        )

        f.write(
            f"Total full data values: "
            f"{split_data.size}\n"
        )

        f.write(
            f"Total MODE input values: "
            f"{offline_data.size}\n"
        )

        # ========================================================
        # MODE timing
        # ========================================================

        f.write(
            "\nMODE TIMING BREAKDOWN [s]\n"
        )

        f.write("-" * 72 + "\n")

        f.write(
            f"Total MODE: "
            f"{timing['total_mode_sec']:.10f}\n"
        )

        f.write(
            f"Clustering: "
            f"{timing['clustering_sec']:.10f}\n"
        )

        f.write(
            f"Training: "
            f"{timing['training_sec']:.10f}\n"
        )

        f.write(
            f"Validation: "
            f"{timing['validation_sec']:.10f}\n"
        )

        f.write(
            f"Stability: "
            f"{timing['stability_sec']:.10f}\n"
        )

        f.write(
            f"Other / NSGA-II: "
            f"{timing['other_sec']:.10f}\n"
        )

        f.write(
            f"Measured components: "
            f"{timing['measured_components_sec']:.10f}\n"
        )

        # ========================================================
        # Energy / outer timer
        # ========================================================

        f.write(
            "\nENERGY / WRAPPER TIMING\n"
        )

        f.write("-" * 72 + "\n")

        f.write(
            f"Tracked duration [s]: "
            f"{initial_energy['duration_sec']:.10f}\n"
        )

        f.write(
            f"Energy [Wh]: "
            f"{initial_energy['energy_wh']:.10f}\n"
        )

        # ========================================================
        # Execution counts
        # ========================================================

        f.write(
            "\nEXECUTION COUNTS\n"
        )

        f.write("-" * 72 + "\n")

        f.write(
            f"Clusterings: "
            f"{timing['n_clusterings']}\n"
        )

        f.write(
            f"Training runs: "
            f"{timing['n_trainings']}\n"
        )

        f.write(
            f"Validation runs: "
            f"{timing['n_validations']}\n"
        )

        f.write(
            f"Stability evaluations: "
            f"{timing['n_stability_evals']}\n"
        )

        f.write(
            f"Clustering cache hits: "
            f"{timing['clustering_cache_hits']}\n"
        )

        # ========================================================
        # Clustering time per K
        # ========================================================

        f.write(
            "\nCLUSTERING TIME BY K [s]\n"
        )

        f.write("-" * 72 + "\n")

        for k, duration in sorted(
            timing[
                "clustering_by_k_sec"
            ].items()
        ):

            f.write(
                f"K={k}: "
                f"{duration:.10f}\n"
            )

        # ========================================================
        # Selected solution
        # ========================================================

        f.write(
            "\nSELECTED MODE CONFIGURATION\n"
        )

        f.write("-" * 72 + "\n")

        for key, value in (
            best_chromosome.items()
        ):

            f.write(
                f"{key}: {value}\n"
            )

        if (
            "best_F"
            in initial_info
        ):

            f.write(
                "Selected objectives: "
                f"{np.asarray(initial_info['best_F']).tolist()}\n"
            )

        # ========================================================
        # Raw RMSE values
        # ========================================================

        f.write(
            "\nRAW-SCALE RMSE FOR EACH "
            "ONLINE TIME SERIES\n"
        )

        f.write("=" * 72 + "\n")

        for row in online_results:

            f.write(
                f"Series "
                f"{row['series_id']}: "
                f"{row['rmse_raw']:.10f}\n"
            )

        # ========================================================
        # RMSE summary
        # ========================================================

        f.write(
            "\nRAW-SCALE RMSE SUMMARY\n"
        )

        f.write("=" * 72 + "\n")

        f.write(
            f"Number of time series: "
            f"{len(raw_rmse_values)}\n"
        )

        f.write(
            f"Mean RMSE: "
            f"{np.mean(raw_rmse_values):.10f}\n"
        )

        f.write(
            f"Std RMSE: "
            f"{np.std(raw_rmse_values, ddof=1):.10f}\n"
        )

        f.write(
            f"Median RMSE: "
            f"{np.median(raw_rmse_values):.10f}\n"
        )

        f.write(
            f"Min RMSE: "
            f"{np.min(raw_rmse_values):.10f}\n"
        )

        f.write(
            f"Max RMSE: "
            f"{np.max(raw_rmse_values):.10f}\n"
        )

    # ============================================================
    # Console summary
    # ============================================================

    print(
        "\nCompleted:",
        file_name,
    )

    print(
        "Total MODE [s]:",
        timing["total_mode_sec"],
    )

    print(
        "Clustering [s]:",
        timing["clustering_sec"],
    )

    print(
        "Mean raw RMSE:",
        np.mean(raw_rmse_values),
    )

    print(
        "Saved:",
        txt_path,
    )

    # ============================================================
    # Return summary
    # ============================================================

    result = {

        "part":
            experiment_part,

        "scale_value":
            scale_value,

        "n_series":
            int(n_series),

        "series_length":
            int(full_series_length),

        "offline_length":
            int(
                offline_data.shape[1]
            ),

        "online_length":
            int(
                online_data.shape[1]
            ),

        "input_values":
            int(
                offline_data.size
            ),

        "total_mode_sec":
            float(
                timing[
                    "total_mode_sec"
                ]
            ),

        "clustering_sec":
            float(
                timing[
                    "clustering_sec"
                ]
            ),

        "training_sec":
            float(
                timing[
                    "training_sec"
                ]
            ),

        "validation_sec":
            float(
                timing[
                    "validation_sec"
                ]
            ),

        "stability_sec":
            float(
                timing[
                    "stability_sec"
                ]
            ),

        "other_sec":
            float(
                timing[
                    "other_sec"
                ]
            ),

        "energy_wh":
            float(
                initial_energy[
                    "energy_wh"
                ]
            ),

        "mean_raw_rmse":
            float(
                np.mean(
                    raw_rmse_values
                )
            ),

        "std_raw_rmse":
            float(
                np.std(
                    raw_rmse_values,
                    ddof=1,
                )
            ),

        "best_k":
            int(
                best_chromosome[
                    "num_clusters"
                ]
            ),

        "txt_path":
            txt_path,
    }

    # Release GPU
    initial_instance.model = (
        initial_instance.model.to("cpu")
    )

    del initial_model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

# PART A: 
## L=500 fixed and vary: N={10,100,500,1000,10000}.

In [ ]:
PART_A_SERIES_LENGTH = 500

PART_A_N_VALUES = [
    10,
    100,
    500,
    1000,
    10000,
]


part_a_folder = os.path.join(
    OUTPUT_ROOT,
    "Part_A_Number_of_Series",
)

os.makedirs(
    part_a_folder,
    exist_ok=True,
)


# ============================================================
# Load ALL available complete series of length 500
# ============================================================

part_a_all_data = (
    AS.read_raspberry_pi_dataset(
        series_length=PART_A_SERIES_LENGTH,
        column=TYPE_OF_DATA,
    )
)


print(
    "Available complete time series:",
    len(part_a_all_data),
)

print(
    "Maximum requested:",
    max(PART_A_N_VALUES),
)

In [ ]:
if len(part_a_all_data) < max(
    PART_A_N_VALUES
):

    print(
        "\nWARNING:"
    )

    print(
        f"You requested up to "
        f"{max(PART_A_N_VALUES)} time series, "
        f"but only {len(part_a_all_data)} "
        f"complete series of length "
        f"{PART_A_SERIES_LENGTH} are available."
    )

In [ ]:
part_a_results = []

for n_series in PART_A_N_VALUES:

    if len(
        part_a_all_data
    ) < n_series:

        print(
            f"\nSkipping N={n_series}: "
            f"only {len(part_a_all_data)} "
            f"series are available."
        )

        continue

    split_data = (
        part_a_all_data[
            :n_series
        ].copy()
    )

    result = (
        run_mode_scalability_case(

            split_data=split_data,

            experiment_part="A",

            scale_value=(
                f"N_{n_series}"
            ),

            full_series_length=(
                PART_A_SERIES_LENGTH
            ),

            output_folder=(
                part_a_folder
            ),
        )
    )

    part_a_results.append(
        result
    )

    # Save summary after EACH successful run.
    # Therefore an interruption at N=10000
    # does not lose previous results.
    pd.DataFrame(
        part_a_results
    ).to_csv(
        os.path.join(
            part_a_folder,
            "Part_A_summary.csv",
        ),
        index=False,
    )

# PART B

## N = 60 (fixed) and L∈{50,100,1000,10000}.

In [ ]:
PART_B_N_SERIES = 60

PART_B_LENGTH_VALUES = [
    50,
    100,
    # 500 already done
    1000,
    10000,
]


part_b_folder = os.path.join(
    OUTPUT_ROOT,
    "Part_B_Series_Length",
)

os.makedirs(
    part_b_folder,
    exist_ok=True,
)

In [ ]:
part_b_results = []

for series_length in (
    PART_B_LENGTH_VALUES
):

    print(
        "\n" + "=" * 80
    )

    print(
        f"Preparing Part B, "
        f"L={series_length}"
    )

    print("=" * 80)

    # ============================================================
    # Read data regrouped into this series length
    # ============================================================

    all_data_this_length = (
        AS.read_raspberry_pi_dataset(

            series_length=(
                series_length
            ),

            column=TYPE_OF_DATA,
        )
    )

    available_n = len(
        all_data_this_length
    )

    print(
        "Available time series:",
        available_n,
    )

    # ============================================================
    # Need exactly 60
    # ============================================================

    if available_n < (
        PART_B_N_SERIES
    ):

        print(
            f"Skipping L={series_length}: "
            f"need {PART_B_N_SERIES} "
            f"time series but only "
            f"{available_n} are available."
        )

        continue

    split_data = (
        all_data_this_length[
            :PART_B_N_SERIES
        ].copy()
    )

    # ============================================================
    # Run MODE
    # ============================================================

    result = (
        run_mode_scalability_case(

            split_data=split_data,

            experiment_part="B",

            scale_value=(
                f"L_{series_length}"
            ),

            full_series_length=(
                series_length
            ),

            output_folder=(
                part_b_folder
            ),
        )
    )

    part_b_results.append(
        result
    )

    # Save after every completed scale.
    pd.DataFrame(
        part_b_results
    ).to_csv(
        os.path.join(
            part_b_folder,
            "Part_B_summary.csv",
        ),
        index=False,
    )

# Inspect summaries

In [ ]:
part_a_summary_df = pd.DataFrame(
    part_a_results
)

part_b_summary_df = pd.DataFrame(
    part_b_results
)

print("\nPART A")
display(
    part_a_summary_df
)

print("\nPART B")
display(
    part_b_summary_df
)